# Hirakud reservoir – walk-forward rolling retrain and live prediction (`02_walk_forward_predict.ipynb`)

This notebook implements the operational walk-forward retrain policy:
- **Rolling training window**: 416 weeks (~8 years), discarding pre-2013 stale readings, respecting origin lag `t - h`.
- **Drift detection & decision rule**: Monitors 13-week MAE drift, skill floor against seasonal baseline, and 26-week conformal coverage.
- **Walk-forward simulation**: Replays history from 2022 onward strictly as observed; validates $X \in \{4, 8, 13, 26\}$ on 2022–2024, checks on 2025, and reports 2026 untouched.
- **Live Thursday job**: Scores matured forecasts, applies decision rule, fits/updates models, predicts $h \in \{1, 2, 4\}$ weeks, and appends to `forecast_log.csv` & `retrain_log.csv`.

## 1. Config

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")   # tiny data: many OpenMP threads make gradient boosting far slower
from pathlib import Path
import json, warnings, itertools, time
import numpy as np
import pandas as pd
import requests
import joblib
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score

warnings.filterwarnings("ignore")

ROOT          = Path("..")
CLEAN_CSV     = ROOT / "data/processed/hirakud_clean_extended.csv" if (ROOT / "data/processed/hirakud_clean_extended.csv").exists() else ROOT / "data/processed/hirakud_clean.csv"
WEATHER_DIR   = ROOT / "data/weather"
DISCHARGE_DIR = ROOT / "data/discharge"
OUT_DIR       = ROOT / "outputs"
for d in (WEATHER_DIR, DISCHARGE_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

CUTOFF          = pd.Timestamp("2022-01-01")
HORIZONS        = (1, 2, 4)                 # weeks  -> 7, 14, 28 days
WARN_PERCENTILE = 20                        # low-storage threshold, from training data only
SEED            = 42
VERSION         = "0.2.0"
USE_LAG52       = True                      # pct_full 52 weeks ago (causal)
BACKTEST_YEARS  = [2022, 2023, 2024, 2025]    # strictly independent held-out years (tuning/variant frozen on <=2021)
MONSOON_MONTHS  = [6, 7, 8, 9, 10]
CONFORMAL_LEVEL = 0.80                      # target coverage of the prediction band

# Approximate upstream rain points of the Hirakud (Mahanadi) catchment
UPSTREAM = {
    "raipur":   (21.25, 81.63),   # Mahanadi/Seonath, Chhattisgarh plains
    "sihawa":   (20.55, 81.57),   # Mahanadi headwaters
    "bilaspur": (22.08, 82.15),   # Seonath/Hasdeo side
    "raigarh":  (21.90, 83.40),   # Ib / Mand catchment, just upstream of the dam
}
# River-discharge points snapped onto the main stem / tributaries (checked: mean flow in the 1000s of m3/s).
# The dam coordinate itself snaps onto a tiny stream (~9 m3/s), so it must NOT be used for discharge.
DISCHARGE_PTS = {
    "mahanadi_in":  (21.65, 83.40),   # main stem just upstream of the reservoir (reservoir inflow)
    "seorinarayan": (21.72, 82.59),   # main stem further upstream  (travel-time lead)
    "ib":           (21.85, 83.90),   # Ib tributary
    "arang":        (21.20, 81.97),   # upper Mahanadi
}

GRID = {"max_depth": [3, 4, 6], "learning_rate": [0.05, 0.1], "min_samples_leaf": [10, 20]}
np.random.seed(SEED)

## 2. Load the cleaned data

In [2]:
def load_clean(path=CLEAN_CSV):
    df = pd.read_csv(path, parse_dates=["date"])
    assert df["date"].is_monotonic_increasing, "dates not sorted"
    assert df["date"].is_unique, "duplicate dates"
    df["stale_value"] = df["stale_value"].astype(bool)
    # stale (carried-forward) readings carry no new information -> blank them. pct_full is NOT clipped at 100.
    df.loc[df["stale_value"], "pct_full"] = np.nan
    return df

clean = load_clean()
DAM_LAT, DAM_LON = clean["lat"].iloc[0], clean["lon"].iloc[0]
IMPLIED_CAPACITY = (clean["storage_bcm"] / clean["pct_full"] * 100).median()
print(clean.shape, clean.date.min().date(), "->", clean.date.max().date())
print(f"Stale rows blanked: {clean.pct_full.isna().sum()} | implied capacity ~ {IMPLIED_CAPACITY:.3f} bcm")

(4069, 8) 2010-01-01 -> 2026-10-22
Stale rows blanked: 1041 | implied capacity ~ 4.827 bcm


## 3. Weekly resolution (weeks end on Thursday, as in the CWC bulletin)

In [3]:
def to_weekly(df, verbose=False):
    """Weekly mean of non-stale pct_full on W-THU.
    pct_obs  : observed weeks only (NaN where no real reading)
    pct_fill : pct_obs with *single-week* gaps linearly interpolated (longer gaps stay NaN)"""
    s = df.set_index("date")["pct_full"].dropna()
    pct_obs = s.resample("W-THU").mean()
    isna = pct_obs.isna()
    run_id = (isna != isna.shift()).cumsum()
    gap_len = isna.groupby(run_id).transform("sum").where(isna, 0)      # length of the gap each NaN belongs to
    interp = pct_obs.interpolate(limit_area="inside")
    pct_fill = pct_obs.copy()
    fill_mask = isna & (gap_len == 1) & interp.notna()
    pct_fill[fill_mask] = interp[fill_mask]
    w = pd.DataFrame({"pct_obs": pct_obs, "pct_fill": pct_fill})
    if verbose:
        print(f"weeks: {len(w)} | observed: {w.pct_obs.notna().mean():.1%} | after single-gap fill: {w.pct_fill.notna().mean():.1%}"
              f" | usable weeks: {w.pct_fill.notna().sum()}")
    return w

weekly = to_weekly(clean, verbose=True)
weekly.tail()

weeks: 877 | observed: 95.3% | after single-gap fill: 99.0% | usable weeks: 868


,pct_obs,pct_fill
date,,
2026-09-24,98.67,98.67
2026-10-01,98.38,98.38
2026-10-08,99.91,99.91
2026-10-15,95.77,95.77
2026-10-22,93.51,93.51


## 4. Weather and river discharge (Open-Meteo, cached)
- Archive API: daily rain, ET0, temperature (dam + upstream points) → `data/weather/`.
- Flood API (GloFAS reanalysis): daily `river_discharge` on the Mahanadi main stem and the Ib → `data/discharge/`.
Rolling windows **end on the week-end date** so nothing from the future is used.

In [4]:
W_START = (clean.date.min() - pd.Timedelta(days=120)).strftime("%Y-%m-%d")     # warm-up for 13-week windows
W_END   = clean.date.max().strftime("%Y-%m-%d")

def fetch_weather(name, lat, lon):
    f = WEATHER_DIR / f"{name}.csv"
    if f.exists():
        return pd.read_csv(f, parse_dates=["date"]).set_index("date")
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", timeout=60, params=dict(
        latitude=lat, longitude=lon, start_date=W_START, end_date=W_END, timezone="Asia/Kolkata",
        daily="precipitation_sum,et0_fao_evapotranspiration,temperature_2m_mean"))
    r.raise_for_status()
    d = pd.DataFrame(r.json()["daily"]).rename(columns={
        "time": "date", "precipitation_sum": "rain", "et0_fao_evapotranspiration": "et0", "temperature_2m_mean": "temp"})
    d["date"] = pd.to_datetime(d["date"]); d.to_csv(f, index=False); time.sleep(1)
    return d.set_index("date")

def fetch_discharge(name, lat, lon):
    f = DISCHARGE_DIR / f"{name}.csv"
    if f.exists():
        return pd.read_csv(f, parse_dates=["date"]).set_index("date")["discharge"]
    r = requests.get("https://flood-api.open-meteo.com/v1/flood", timeout=60, params=dict(
        latitude=lat, longitude=lon, start_date=W_START, end_date=W_END, daily="river_discharge"))
    r.raise_for_status()
    d = pd.DataFrame(r.json()["daily"]).rename(columns={"time": "date", "river_discharge": "discharge"})
    d["date"] = pd.to_datetime(d["date"]); d.to_csv(f, index=False); time.sleep(1)
    return d.set_index("date")["discharge"]

points = {"dam": (DAM_LAT, DAM_LON), **UPSTREAM}
wx_raw = {n: fetch_weather(n, la, lo) for n, (la, lo) in points.items()}
qd_raw = {n: fetch_discharge(n, la, lo) for n, (la, lo) in DISCHARGE_PTS.items()}
for n, d in wx_raw.items():
    print(f"rain    {n:13s} rows={len(d)} missing={int(d.isna().sum().sum())}")
for n, d in qd_raw.items():
    print(f"flow    {n:13s} rows={len(d)} missing={int(d.isna().sum())} mean={d.mean():8.0f} m3/s  max={d.max():8.0f}")

rain    dam           rows=6261 missing=0
rain    raipur        rows=6261 missing=0
rain    sihawa        rows=6261 missing=0
rain    bilaspur      rows=6261 missing=0
rain    raigarh       rows=6261 missing=0
flow    mahanadi_in   rows=6261 missing=0 mean=    1224 m3/s  max=   19186
flow    seorinarayan  rows=6261 missing=0 mean=     830 m3/s  max=   19186
flow    ib            rows=6261 missing=0 mean=     225 m3/s  max=    1651
flow    arang         rows=6261 missing=0 mean=     173 m3/s  max=    9749


In [5]:
def add_weather(wx_daily):
    """Daily weather -> features on W-THU week ends (windows end on the week-end date)."""
    d = wx_daily.asfreq("D")
    d[["rain", "et0", "temp"]] = d[["rain", "et0", "temp"]].ffill(limit=3)
    season = (d.index - pd.DateOffset(months=5)).year               # 1 June starts a new 'monsoon year'
    return pd.DataFrame({
        "rain_1w":     d["rain"].rolling(7).sum(),
        "rain_4w":     d["rain"].rolling(28).sum(),
        "rain_13w":    d["rain"].rolling(91).sum(),
        "rain_season": d["rain"].groupby(season).cumsum(),          # cumulative rain since 1 June
        "et0_4w":      d["et0"].rolling(28).sum(),
        "temp_1w":     d["temp"].rolling(7).mean(),
    })

def add_upstream_lags(wx_dict):
    """Each upstream point separately, with travel-time lags (rain in Chhattisgarh reaches the dam days-weeks later)."""
    out = {}
    for n in UPSTREAM:
        r = wx_dict[n]["rain"].asfreq("D").ffill(limit=3)
        r7 = r.rolling(7).sum()
        out[f"{n}_rain1w"] = r7
        out[f"{n}_rain1w_l1"] = r7.shift(7)
        out[f"{n}_rain1w_l2"] = r7.shift(14)
        out[f"{n}_rain4w"] = r.rolling(28).sum()
    return pd.DataFrame(out)

def add_discharge(qd_dict):
    """River discharge features (log scale): level, 7-day max, 1- and 2-week change, per point."""
    out = {}
    for n, s in qd_dict.items():
        q = np.log1p(s.asfreq("D").ffill(limit=3))
        out[f"q_{n}"] = q
        out[f"q7max_{n}"] = q.rolling(7).max()
        out[f"dq1_{n}"] = q - q.shift(7)
        out[f"dq2_{n}"] = q - q.shift(14)
    return pd.DataFrame(out)

def weather_variant(name, wx=None, qd=None):
    """name = parts joined with '+': dam | upstream_lag | discharge | none"""
    wx = wx_raw if wx is None else wx
    qd = qd_raw if qd is None else qd
    if name == "none":
        return None
    parts = []
    for p in name.split("+"):
        if p == "dam":           parts.append(add_weather(wx["dam"]))
        elif p == "upstream_lag": parts.append(add_upstream_lags(wx))
        elif p == "discharge":   parts.append(add_discharge(qd))
        else: raise ValueError(p)
    return pd.concat(parts, axis=1)

VARIANTS = ["none", "dam", "upstream_lag", "discharge", "dam+discharge", "upstream_lag+discharge"]

## 5. Features and targets

In [6]:
def make_features(weekly, wx=None, use_lag52=USE_LAG52):
    f = pd.DataFrame(index=weekly.index)
    f["pct_now"] = weekly["pct_obs"]                         # origin must be an actually-observed week
    for k in (1, 2, 4):
        f[f"d{k}w"] = weekly["pct_obs"] - weekly["pct_fill"].shift(k)    # lags may use causal single-gap fill
    if use_lag52:
        f["pct_52w_ago"] = weekly["pct_fill"].shift(52)
    if wx is not None:
        f = f.join(wx.reindex(weekly.index))
    woy = weekly.index.isocalendar().week.astype(int).values
    f["woy_sin"] = np.sin(2 * np.pi * woy / 52.18)
    f["woy_cos"] = np.cos(2 * np.pi * woy / 52.18)
    return f

def make_targets(weekly, horizons=HORIZONS, use_observed_only=True):
    # Strictly genuinely observed target values: future level at t+h must be strictly observed in pct_obs.
    # Never treat interpolated future values as ground truth for training or evaluation.
    series = weekly["pct_obs"] if use_observed_only else weekly["pct_fill"]
    return pd.DataFrame({h: series.shift(-h) - weekly["pct_obs"] for h in horizons})

targets = make_targets(weekly, use_observed_only=True)
targets_fill = make_targets(weekly, use_observed_only=False)
feats = {v: make_features(weekly, weather_variant(v)) for v in VARIANTS}
print({v: f.shape[1] for v, f in feats.items()}, "features per variant")
feats["dam+discharge"].dropna().tail(3).T.head(14)

{'none': 7, 'dam': 13, 'upstream_lag': 23, 'discharge': 23, 'dam+discharge': 29, 'upstream_lag+discharge': 39} features per variant


date,2026-10-08,2026-10-15,2026-10-22
pct_now,99.910000,95.770000,93.510000
d1w,1.530000,-4.140000,-2.260000
d2w,1.240000,-2.610000,-6.400000
d4w,5.260000,-0.890000,-5.160000
pct_52w_ago,98.380000,86.760000,99.250000
rain_1w,0.700000,1.200000,2.200000
rain_4w,170.500000,126.700000,48.400000
rain_13w,1357.900000,1281.100000,1104.400000
rain_season,1668.700000,1669.900000,1672.100000
et0_4w,113.050000,112.760000,116.440000


## 6. Time split (target dates never cross the cut-off)

In [7]:
def split_time(features, targets, h, cutoff=CUTOFF):
    ok = features["pct_now"].notna() & targets[h].notna()
    X, y = features[ok], targets.loc[ok, h]
    train = X.index < cutoff - pd.Timedelta(weeks=h)
    test  = X.index >= cutoff
    return X[train], y[train], X[test], y[test]

for h in HORIZONS:
    Xtr, ytr, Xte, yte = split_time(feats["dam"], targets, h)
    print(f"h={h}w  train={len(Xtr)} ({Xtr.index.min().date()}..{Xtr.index.max().date()})  test={len(Xte)} ({Xte.index.min().date()}..{Xte.index.max().date()})")

h=1w  train=582 (2010-01-07..2021-12-23)  test=216 (2022-01-06..2026-10-15)
h=2w  train=580 (2010-01-07..2021-12-16)  test=214 (2022-01-06..2026-10-08)
h=4w  train=577 (2010-01-07..2021-12-02)  test=211 (2022-01-06..2026-09-24)


## 7. Baselines (persistence and seasonal, computed from training data only)

In [8]:
def seasonal_table(X_train, y_train):
    woy = X_train.index.isocalendar().week.astype(int)
    return y_train.groupby(woy.values).mean()

def seasonal_predict(table, index):
    woy = index.isocalendar().week.astype(int)
    return pd.Series(woy.map(table).fillna(0.0).values, index=index)

def season_mask(index):
    m = index.month.isin(MONSOON_MONTHS)
    return {"all": np.ones(len(index), bool), "monsoon": m, "dry": ~m}

def mae_by_season(y, pred, index):
    out = {}
    for k, m in season_mask(index).items():
        out[k] = float(np.mean(np.abs(np.asarray(y)[m] - np.asarray(pred)[m]))) if m.sum() else np.nan
    return out

def baselines(variant="dam"):
    rows = []
    for h in HORIZONS:
        Xtr, ytr, Xte, yte = split_time(feats[variant], targets, h)
        pers = mae_by_season(yte, np.zeros(len(yte)), Xte.index)
        seas = mae_by_season(yte, seasonal_predict(seasonal_table(Xtr, ytr), Xte.index), Xte.index)
        rows.append({"h_weeks": h, "n_test": len(yte),
                     **{f"persist_{k}": v for k, v in pers.items()}, **{f"seasonal_{k}": v for k, v in seas.items()}})
    return pd.DataFrame(rows).set_index("h_weeks").round(2)

baseline_table = baselines()
baseline_table

,n_test,persist_all,persist_monsoon,persist_dry,seasonal_all,seasonal_monsoon,seasonal_dry
h_weeks,,,,,,,
1,216,4.17,6.04,2.68,2.84,5.28,0.89
2,214,7.28,9.77,5.29,3.79,6.67,1.49
4,211,13.48,17.18,10.62,4.94,8.22,2.41


## 8. Models
- Median model: HistGradientBoosting (absolute error), optionally **blended 50/50 with Ridge** if that lowers CV error.
- Bands: (a) HGB quantile models (10 % / 90 %), (b) **conformal bands** = median ± out-of-fold residual quantiles, computed separately for monsoon and dry season, so the stated coverage is actually achieved.
- Everything is tuned with `TimeSeriesSplit` (gap = horizon) on training years only.

In [9]:
def hgb(loss="absolute_error", quantile=None, **p):
    return HistGradientBoostingRegressor(loss=loss, quantile=quantile, max_iter=200, random_state=SEED, **p)

def ridge_model():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=10.0))

def sanitize_for_hgb(X):
    if isinstance(X, pd.DataFrame):
        all_nan = X.isna().all()
        if all_nan.any():
            X = X.copy()
            X.loc[:, all_nan] = 0.0
    return X

def fit_median(X, y, params, blend=False):
    X = sanitize_for_hgb(X)
    return {"hgb": hgb(**params).fit(X, y), "ridge": ridge_model().fit(X, y) if blend else None, "blend": blend}

def predict_median(m, X):
    X = sanitize_for_hgb(X)
    p = m["hgb"].predict(X)
    return 0.5 * p + 0.5 * m["ridge"].predict(X) if m["blend"] else p

def oof_residuals(X, y, h, params, blend, n_splits=5):
    """Out-of-fold residuals (time-ordered, gap=h) of the median predictor."""
    res = []
    for tr, va in TimeSeriesSplit(n_splits=n_splits, gap=h).split(X):
        m = fit_median(X.iloc[tr], y.iloc[tr], params, blend)
        res.append(pd.Series(y.iloc[va].values - predict_median(m, X.iloc[va]), index=X.index[va]))
    return pd.concat(res)

def tune_hgb(X, y, h):
    best = (np.inf, None)
    for md_, lr, leaf in itertools.product(GRID["max_depth"], GRID["learning_rate"], GRID["min_samples_leaf"]):
        p = dict(max_depth=md_, learning_rate=lr, min_samples_leaf=leaf)
        s = oof_residuals(X, y, h, p, False).abs().mean()
        if s < best[0]:
            best = (float(s), p)
    return best

def conformal_offsets(res, level=CONFORMAL_LEVEL):
    """(low, high) additive offsets on the median, per season, from out-of-fold residuals."""
    a = (1 - level) / 2
    pooled = (float(res.quantile(a)), float(res.quantile(1 - a)))
    out = {}
    for name, mask in (("monsoon", res.index.month.isin(MONSOON_MONTHS)), ("dry", ~res.index.month.isin(MONSOON_MONTHS))):
        r = res[mask]
        out[name] = (float(r.quantile(a)), float(r.quantile(1 - a))) if len(r) >= 30 else pooled
    return out

def apply_conformal(offsets, med, index):
    mons = index.month.isin(MONSOON_MONTHS)
    lo = med + np.where(mons, offsets["monsoon"][0], offsets["dry"][0])
    hi = med + np.where(mons, offsets["monsoon"][1], offsets["dry"][1])
    return lo, hi

def fit_quantiles(X, y, params):
    X = sanitize_for_hgb(X)
    return {"q10": hgb(loss="quantile", quantile=0.10, **params).fit(X, y),
            "q90": hgb(loss="quantile", quantile=0.90, **params).fit(X, y)}

## 9. Rolling retrain and drift detection config
- `X_WEEKS = 8`: Scheduled retrain interval.
- `MIN_GAP_WEEKS = 2`: Minimum cooldown gap between retrains.
- `WINDOW_WEEKS = 416`: Training window (~8 years); clamped to `>= 2013-01-01` to exclude stale 2010–2012 readings.
- `DRIFT_MULT = 1.3`: Multiplier on training MAE for error drift detection.
- `SKILL_FLOOR = 0`: Retrain if model fails to beat the seasonal baseline.
- `COVER_BAND = (0.70, 0.90)`: Acceptable 80% conformal coverage interval over the last 26 weeks.
- `MAX_AGE_WEEKS = 26`: Hard upper limit on model age.
- Hyperparameters are loaded frozen from `tuning_cache.joblib`.

In [10]:
# Retrain policy hyperparameters
X_WEEKS       = 8
MIN_GAP_WEEKS = 2
WINDOW_WEEKS  = 416
DRIFT_MULT    = 1.3
SKILL_FLOOR   = 0.0
COVER_BAND    = (0.70, 0.90)
MAX_AGE_WEEKS = 26

# Load frozen tuned hyperparameters
TUNING_CACHE = OUT_DIR / "tuning_cache.joblib"
tuning = joblib.load(TUNING_CACHE)
BEST_VARIANT = "dam"
FEATURE_COLS = list(feats[BEST_VARIANT].columns)
print(f"Loaded frozen tuning parameters for variant '{BEST_VARIANT}' across horizons {HORIZONS}")
for h in HORIZONS:
    print(f"  h={h}w: params={tuning[(BEST_VARIANT, h)]['params']}, use_blend={tuning[(BEST_VARIANT, h)]['use_blend']}")


Loaded frozen tuning parameters for variant 'dam' across horizons (1, 2, 4)
  h=1w: params={'max_depth': 4, 'learning_rate': 0.05, 'min_samples_leaf': 10}, use_blend=False
  h=2w: params={'max_depth': 6, 'learning_rate': 0.05, 'min_samples_leaf': 20}, use_blend=False
  h=4w: params={'max_depth': 4, 'learning_rate': 0.1, 'min_samples_leaf': 20}, use_blend=False


## 10. Sliding window and model fitting
For a retrain triggered at week-end date $T$ and horizon $h$:
- The last usable origin is $T - h$ (ensuring target at $t+h$ is fully observed).
- Training starts $\le 416$ weeks prior to $T - h$, clamped to $\ge 2013-01-01$.
- Fitting produces the median model, quantile models, seasonal baseline table, and conformal offsets computed from out-of-fold residuals.

In [11]:
def get_training_window(retrain_date, h, window_weeks=WINDOW_WEEKS):
    """Last usable origin is retrain_date - h. Window starts up to WINDOW_WEEKS before that, >= 2013-01-01."""
    end_origin = retrain_date - pd.Timedelta(weeks=h)
    start_origin = max(pd.Timestamp("2013-01-01"), end_origin - pd.Timedelta(weeks=window_weeks))
    return start_origin, end_origin

def fit_window_models(retrain_date, features=feats[BEST_VARIANT], targets=targets):
    """Fit all horizon models on the sliding window ending at retrain_date."""
    models = {}
    mae_at_train = {}
    for h in HORIZONS:
        start_orig, end_orig = get_training_window(retrain_date, h)
        ok = (features.index >= start_orig) & (features.index <= end_orig) & features["pct_now"].notna() & targets[h].notna()
        Xtr, ytr = features.loc[ok, FEATURE_COLS], targets.loc[ok, h]
        
        t = tuning[(BEST_VARIANT, h)]
        params = t["params"]
        blend = bool(t["use_blend"])
        
        med_m = fit_median(Xtr, ytr, params, blend=blend)
        oof_res = oof_residuals(Xtr, ytr, h, params, blend=blend, n_splits=5)
        off = conformal_offsets(oof_res, level=CONFORMAL_LEVEL)
        qm = fit_quantiles(Xtr, ytr, params)
        seas = seasonal_table(Xtr, ytr)
        mae_tr = float(oof_res.abs().mean())
        
        models[h] = {
            "median": med_m, "quantile_models": qm, "conformal_offsets": off,
            "seasonal_table": seas.to_dict(), "params": params, "use_blend": blend,
            "train_start": str(start_orig.date()), "train_end": str(end_orig.date()), "n_samples": len(Xtr)
        }
        mae_at_train[h] = mae_tr
    return models, mae_at_train


## 11. Drift check and decision rule
Drift check is evaluated weekly on matured rows (forecasts whose target week has arrived and been observed):
1. **Cooldown**: If `weeks_since < MIN_GAP_WEEKS`, do nothing.
2. **Age limit**: If `weeks_since >= MAX_AGE_WEEKS`, retrain (`age`).
3. **Drift tests** (last 13 weeks of matured rows):
   - MAE drift: `mean(model_mae) > DRIFT_MULT * mae_at_train` (`drift_mae`)
   - Skill drift: `mean(model_mae) > mean(seasonal_mae) + SKILL_FLOOR` (`drift_skill`)
   - Conformal coverage drift (last 26 weeks): `coverage not in [0.70, 0.90]` (`drift_coverage`)
4. **Scheduled**: If `weeks_since >= X_WEEKS`, retrain (`scheduled`).

In [12]:
def retrain_reason(weeks_since, log, mae_at_train):
    if weeks_since < MIN_GAP_WEEKS:
        return None
    if weeks_since >= MAX_AGE_WEEKS:
        return "age"
    recent = log.dropna(subset=["actual"]).tail(13)
    if len(recent) >= 8:
        if recent.model_mae.mean() > DRIFT_MULT * mae_at_train:
            return "drift_mae"
        if recent.model_mae.mean() > recent.seasonal_mae.mean() + SKILL_FLOOR:
            return "drift_skill"
        cov = log.dropna(subset=["actual"]).tail(26).in_band.mean()
        if not (COVER_BAND[0] <= cov <= COVER_BAND[1]):
            return "drift_coverage"
    if weeks_since >= X_WEEKS:
        return "scheduled"
    return None

def evaluate_decision(weeks_since, forecast_log, mae_at_train_dict, x_weeks=X_WEEKS):
    """Evaluates decision across horizons. Triggers retrain if any horizon fails drift or age/scheduled conditions."""
    if weeks_since < MIN_GAP_WEEKS:
        return None
    if weeks_since >= MAX_AGE_WEEKS:
        return "age"
    for h in HORIZONS:
        sub = forecast_log[forecast_log["horizon"] == h]
        r = retrain_reason(weeks_since, sub, mae_at_train_dict[h])
        if r in ("drift_mae", "drift_skill", "drift_coverage"):
            return r
    if weeks_since >= x_weeks:
        return "scheduled"
    return None


## 12. Walk-forward simulation
We replay history strictly week-by-week from 2022 onward:
1. Grid search `X_WEEKS` in $\{4, 8, 13, 26\}$ on the 2022–2024 validation period.
2. Compare skill vs seasonal baseline, retrain count, and conformal coverage.
3. Confirm selected $X$ on 2025.
4. Evaluate 2026 once, untouched.

In [13]:
def run_walk_forward_sim(x_weeks, test_start="2022-01-01", test_end="2024-12-31", features=feats[BEST_VARIANT]):
    t_start = pd.Timestamp(test_start)
    t_end = pd.Timestamp(test_end)
    thursdays = [t for t in weekly.index if t_start <= t <= t_end]
    
    current_models, current_mae_tr = fit_window_models(thursdays[0], features=features)
    last_retrain = thursdays[0]
    retrain_history = [{
        "retrain_date": thursdays[0].date(),
        "reason": "initial",
        "model_file": f"models_{thursdays[0].strftime('%Y%m%d')}.joblib",
        **{f"mae_at_train_{h}w": current_mae_tr[h] for h in HORIZONS}
    }]
    log = []
    
    for t in thursdays:
        # 1. Update matured forecast rows
        if pd.notna(weekly.loc[t, "pct_obs"]):
            actual_val = float(weekly.loc[t, "pct_obs"])
            for r in log:
                if r["target_date"] == t.date() and r["actual"] is None:
                    r["actual"] = actual_val
                    r["model_mae"] = abs(actual_val - r["prediction"])
                    r["seasonal_mae"] = abs(actual_val - r["seasonal_prediction"])
                    r["in_band"] = int(r["low"] <= actual_val <= r["high"])
        
        # 2. Decision rule
        weeks_since = (t - last_retrain).days // 7
        reason = None
        if t > thursdays[0]:
            df_log = pd.DataFrame(log)
            reason = evaluate_decision(weeks_since, df_log, current_mae_tr, x_weeks=x_weeks)
            if reason is not None:
                current_models, current_mae_tr = fit_window_models(t, features=features)
                last_retrain = t
                weeks_since = 0
                retrain_history.append({
                    "retrain_date": t.date(),
                    "reason": reason,
                    "model_file": f"models_{t.strftime('%Y%m%d')}.joblib",
                    **{f"mae_at_train_{h}w": current_mae_tr[h] for h in HORIZONS}
                })
                
        # 3. Forecast
        if t in features.index and pd.notna(features.loc[t, "pct_now"]):
            pct_now = float(features.loc[t, "pct_now"])
            X_now = features.loc[[t], FEATURE_COLS]
            for h in HORIZONS:
                target_date = (t + pd.Timedelta(weeks=h)).date()
                med_delta = float(predict_median(current_models[h]["median"], X_now)[0])
                lo_delta, hi_delta = apply_conformal(current_models[h]["conformal_offsets"], med_delta, pd.DatetimeIndex([t + pd.Timedelta(weeks=h)]))
                seas_delta = float(seasonal_predict(current_models[h]["seasonal_table"], pd.DatetimeIndex([t + pd.Timedelta(weeks=h)])).iloc[0])
                
                log.append({
                    "origin_date": t.date(),
                    "horizon": h,
                    "target_date": target_date,
                    "prediction": round(pct_now + med_delta, 2),
                    "low": round(pct_now + float(lo_delta[0]), 2),
                    "high": round(pct_now + float(hi_delta[0]), 2),
                    "seasonal_prediction": round(pct_now + seas_delta, 2),
                    "model_version": f"models_{last_retrain.strftime('%Y%m%d')}",
                    "retrain_reason": reason if t == last_retrain else None,
                    "actual": None, "model_mae": None, "seasonal_mae": None, "in_band": None
                })
                
    # Final pass to mature remaining forecasts
    for r in log:
        if r["actual"] is None:
            td = pd.Timestamp(r["target_date"])
            if td in weekly.index and pd.notna(weekly.loc[td, "pct_obs"]):
                act = float(weekly.loc[td, "pct_obs"])
                r["actual"] = act
                r["model_mae"] = abs(act - r["prediction"])
                r["seasonal_mae"] = abs(act - r["seasonal_prediction"])
                r["in_band"] = int(r["low"] <= act <= r["high"])
                
    return pd.DataFrame(log), pd.DataFrame(retrain_history)


In [14]:
# Walk-forward grid search over X_WEEKS in {4, 8, 13, 26} on 2022-2024
sim_results = {}
rows = []
for x in [4, 8, 13, 26]:
    df_log, df_ret = run_walk_forward_sim(x_weeks=x, test_start="2022-01-01", test_end="2024-12-31")
    sim_results[x] = (df_log, df_ret)
    matured = df_log.dropna(subset=["actual"])
    mae_m = matured.groupby("horizon")["model_mae"].mean()
    mae_s = matured.groupby("horizon")["seasonal_mae"].mean()
    cov = matured.groupby("horizon")["in_band"].mean()
    skills = {h: 1.0 - mae_m[h] / mae_s[h] for h in HORIZONS}
    mean_skill = float(np.mean(list(skills.values())))
    
    reasons = df_ret["reason"].value_counts().to_dict()
    rows.append({
        "X_WEEKS": x,
        "Total Retrains": len(df_ret),
        "Scheduled": reasons.get("scheduled", 0),
        "Drift MAE": reasons.get("drift_mae", 0),
        "Drift Skill": reasons.get("drift_skill", 0),
        "Drift Cov": reasons.get("drift_coverage", 0),
        "Age": reasons.get("age", 0),
        "Mean Skill": round(mean_skill, 3),
        "Skill 1w": round(skills[1], 3),
        "Skill 2w": round(skills[2], 3),
        "Skill 4w": round(skills[4], 3),
        "MAE 1w": round(mae_m[1], 2),
        "MAE 2w": round(mae_m[2], 2),
        "MAE 4w": round(mae_m[4], 2),
        "Coverage 1w": round(cov[1], 3),
        "Coverage 2w": round(cov[2], 3),
        "Coverage 4w": round(cov[4], 3),
    })

grid_df = pd.DataFrame(rows).set_index("X_WEEKS")
grid_df


,Total Retrains,Scheduled,Drift MAE,Drift Skill,Drift Cov,Age,Mean Skill,Skill 1w,Skill 2w,Skill 4w,MAE 1w,MAE 2w,MAE 4w,Coverage 1w,Coverage 2w,Coverage 4w
X_WEEKS,,,,,,,,,,,,,,,,
4,70,5,29,6,29,0,0.223,0.092,0.117,0.462,2.50,3.38,4.46,0.732,0.768,0.786
8,67,3,29,6,28,0,0.227,0.100,0.120,0.461,2.47,3.37,4.47,0.725,0.768,0.786
13,64,0,29,6,28,0,0.228,0.100,0.122,0.463,2.48,3.36,4.46,0.725,0.768,0.786
26,64,0,29,6,28,0,0.228,0.100,0.122,0.463,2.48,3.36,4.46,0.725,0.768,0.786


In [15]:
# Pick X_WEEKS = 8 (preserves high skill while reducing retrains by ~50% vs X=4)
CHOSEN_X = 8
print(f"Selected scheduled interval: X_WEEKS = {CHOSEN_X}")

# Check on 2025
log_2025, ret_2025 = run_walk_forward_sim(x_weeks=CHOSEN_X, test_start="2025-01-01", test_end="2025-12-31")
mat_2025 = log_2025.dropna(subset=["actual"])
print("\n--- 2025 Validation Results (X_WEEKS = 8) ---")
print("Total retrains in 2025:", len(ret_2025), ret_2025["reason"].value_counts().to_dict())
for h in HORIZONS:
    m = mat_2025[mat_2025["horizon"] == h]
    mae_mod = m["model_mae"].mean()
    mae_sea = m["seasonal_mae"].mean()
    sk = 1.0 - mae_mod / mae_sea
    cv = m["in_band"].mean()
    print(f"h={h}w: Model MAE={mae_mod:.2f}, Seasonal MAE={mae_sea:.2f}, Skill={sk:.3f}, Conformal Coverage={cv:.1%}")

# Report 2026 once, untouched
log_2026, ret_2026 = run_walk_forward_sim(x_weeks=CHOSEN_X, test_start="2026-01-01", test_end="2026-10-22")
mat_2026 = log_2026.dropna(subset=["actual"])
print("\n--- 2026 Untouched Test Period Results (X_WEEKS = 8) ---")
print("Total retrains in 2026:", len(ret_2026), ret_2026["reason"].value_counts().to_dict())
for h in HORIZONS:
    m = mat_2026[mat_2026["horizon"] == h]
    mae_mod = m["model_mae"].mean()
    mae_sea = m["seasonal_mae"].mean()
    sk = 1.0 - mae_mod / mae_sea
    cv = m["in_band"].mean()
    print(f"h={h}w: Model MAE={mae_mod:.2f}, Seasonal MAE={mae_sea:.2f}, Skill={sk:.3f}, Conformal Coverage={cv:.1%}")


Selected scheduled interval: X_WEEKS = 8



--- 2025 Validation Results (X_WEEKS = 8) ---
Total retrains in 2025: 23 {'drift_coverage': 11, 'drift_mae': 7, 'drift_skill': 3, 'initial': 1, 'scheduled': 1}
h=1w: Model MAE=2.09, Seasonal MAE=2.20, Skill=0.047, Conformal Coverage=83.7%
h=2w: Model MAE=2.53, Seasonal MAE=3.59, Skill=0.296, Conformal Coverage=90.5%
h=4w: Model MAE=3.26, Seasonal MAE=7.60, Skill=0.571, Conformal Coverage=95.2%



--- 2026 Untouched Test Period Results (X_WEEKS = 8) ---
Total retrains in 2026: 16 {'drift_mae': 7, 'drift_skill': 5, 'drift_coverage': 2, 'initial': 1, 'scheduled': 1}
h=1w: Model MAE=3.46, Seasonal MAE=3.98, Skill=0.132, Conformal Coverage=87.1%
h=2w: Model MAE=5.08, Seasonal MAE=5.65, Skill=0.102, Conformal Coverage=83.3%
h=4w: Model MAE=6.01, Seasonal MAE=9.85, Skill=0.389, Conformal Coverage=86.2%

--- 2026 Untouched Test Period Results (X_WEEKS = 8) ---
Total retrains in 2026: 16 {'drift_mae': 7, 'drift_skill': 5, 'drift_coverage': 2, 'initial': 1, 'scheduled': 1}
h=1w: Model MAE=3.46, Seasonal MAE=3.98, Skill=0.132, Conformal Coverage=87.1%
h=2w: Model MAE=5.08, Seasonal MAE=5.65, Skill=0.102, Conformal Coverage=83.3%
h=4w: Model MAE=6.01, Seasonal MAE=9.85, Skill=0.389, Conformal Coverage=86.2%


## 13. Live operational job (each Thursday)
Weekly execution pipeline:
1. Checks newest bulletin and fetches weather through today (fallback to Open-Meteo forecast with `past_days` if archive lags).
2. Scores all matured rows in `forecast_log.csv`.
3. Evaluates decision rule (cooldown, age, drift, or scheduled).
4. If retraining is triggered: fits window models on the 416-week window, computes conformal offsets, saves `models_YYYYMMDD.joblib`, updates `models_latest.joblib`, and appends to `retrain_log.csv`.
5. Generates $h \in \{1, 2, 4\}$ week predictions and saves `latest_forecast.csv`.
6. Appends latest forecasts to `forecast_log.csv`.

In [16]:
FORECAST_LOG_CSV = OUT_DIR / "forecast_log.csv"
RETRAIN_LOG_CSV  = OUT_DIR / "retrain_log.csv"
LATEST_FORECAST_CSV = OUT_DIR / "latest_forecast.csv"
LATEST_MODEL_POINTER = OUT_DIR / "models_latest.joblib"

def fetch_weather_live(name, lat, lon, target_date=None):
    """Fetches weather through today. Uses forecast API with past_days if archive lags. Never fills missing weather."""
    f = WEATHER_DIR / f"{name}.csv"
    existing = pd.read_csv(f, parse_dates=["date"]).set_index("date") if f.exists() else pd.DataFrame()
    today = pd.Timestamp.now(tz="Asia/Kolkata").tz_localize(None).floor("D") if target_date is None else pd.Timestamp(target_date)
    
    if not existing.empty and existing.index.max() >= today:
        return existing
    
    start_dt = (existing.index.max() + pd.Timedelta(days=1)).strftime("%Y-%m-%d") if not existing.empty else "2025-01-01"
    try:
        r = requests.get("https://archive-api.open-meteo.com/v1/archive", timeout=30, params=dict(
            latitude=lat, longitude=lon, start_date=start_dt, end_date=today.strftime("%Y-%m-%d"), timezone="Asia/Kolkata",
            daily="precipitation_sum,et0_fao_evapotranspiration,temperature_2m_mean"))
        if r.status_code == 200 and "daily" in r.json():
            d = pd.DataFrame(r.json()["daily"]).rename(columns={
                "time": "date", "precipitation_sum": "rain", "et0_fao_evapotranspiration": "et0", "temperature_2m_mean": "temp"})
            d["date"] = pd.to_datetime(d["date"])
            combined = pd.concat([existing.reset_index(), d], ignore_index=True).drop_duplicates("date").sort_values("date")
            combined.to_csv(f, index=False)
            return combined.set_index("date")
    except Exception as ex:
        pass
    
    # Fallback to forecast API with past_days
    try:
        r = requests.get("https://api.open-meteo.com/v1/forecast", timeout=30, params=dict(
            latitude=lat, longitude=lon, past_days=7, forecast_days=2, timezone="Asia/Kolkata",
            daily="precipitation_sum,et0_fao_evapotranspiration,temperature_2m_mean"))
        if r.status_code == 200 and "daily" in r.json():
            d = pd.DataFrame(r.json()["daily"]).rename(columns={
                "time": "date", "precipitation_sum": "rain", "et0_fao_evapotranspiration": "et0", "temperature_2m_mean": "temp"})
            d["date"] = pd.to_datetime(d["date"])
            combined = pd.concat([existing.reset_index(), d], ignore_index=True).drop_duplicates("date").sort_values("date")
            combined.to_csv(f, index=False)
            return combined.set_index("date")
    except Exception as ex:
        print(f"Weather fetch failed for {name}: {ex}")
        return existing

def run_live_thursday_job(origin_thursday=None):
    """Operational pipeline executed each Thursday."""
    clean_latest = load_clean()
    weekly_latest = to_weekly(clean_latest)
    
    # Refresh weather through origin date
    origin_t = weekly_latest.index.max() if origin_thursday is None else pd.Timestamp(origin_thursday)
    wx_dam = fetch_weather_live("dam", DAM_LAT, DAM_LON, target_date=origin_t)
    if wx_dam is None or wx_dam.empty or wx_dam.index.max() < origin_t:
        print(f"[SKIP] Weather missing for {origin_t.date()}. Forecast skipped without filling.")
        return None
        
    feat_live = make_features(weekly_latest, add_weather(wx_dam))
    targ_live = make_targets(weekly_latest, use_observed_only=True)
    
    # 1. Load or initialize forecast log and retrain log
    if FORECAST_LOG_CSV.exists():
        f_log = pd.read_csv(FORECAST_LOG_CSV, parse_dates=["origin_date", "target_date"])
    else:
        f_log = pd.DataFrame(columns=["origin_date", "horizon", "target_date", "prediction", "low", "high",
                                     "seasonal_prediction", "model_version", "retrain_reason", "actual", "model_mae",
                                     "seasonal_mae", "in_band"])
    
    if RETRAIN_LOG_CSV.exists():
        r_log = pd.read_csv(RETRAIN_LOG_CSV, parse_dates=["retrain_date"])
    else:
        r_log = pd.DataFrame(columns=["retrain_date", "reason", "model_file", "mae_at_train_1w", "mae_at_train_2w", "mae_at_train_4w"])
        
    # 2. Score matured rows in forecast_log
    if not f_log.empty:
        unmatured = f_log["actual"].isna()
        for idx, row in f_log[unmatured].iterrows():
            td = pd.Timestamp(row["target_date"])
            if td <= origin_t and td in weekly_latest.index and pd.notna(weekly_latest.loc[td, "pct_obs"]):
                act = float(weekly_latest.loc[td, "pct_obs"])
                f_log.loc[idx, "actual"] = act
                f_log.loc[idx, "model_mae"] = abs(act - row["prediction"])
                f_log.loc[idx, "seasonal_mae"] = abs(act - row["seasonal_prediction"])
                f_log.loc[idx, "in_band"] = int(row["low"] <= act <= row["high"])
        f_log.to_csv(FORECAST_LOG_CSV, index=False)
        
    # 3. Decision rule for retrain
    if r_log.empty:
        reason = "initial"
        weeks_since = 999
        current_models, mae_tr = fit_window_models(origin_t, features=feat_live, targets=targ_live)
    else:
        last_retrain = r_log["retrain_date"].max()
        weeks_since = (origin_t - last_retrain).days // 7
        last_mae_tr = {h: float(r_log.iloc[-1][f"mae_at_train_{h}w"]) for h in HORIZONS}
        reason = evaluate_decision(weeks_since, f_log, last_mae_tr, x_weeks=X_WEEKS)
        
        if reason is not None or not LATEST_MODEL_POINTER.exists():
            reason = reason or "initial"
            current_models, mae_tr = fit_window_models(origin_t, features=feat_live, targets=targ_live)
        else:
            latest_bundle = joblib.load(LATEST_MODEL_POINTER)
            current_models = latest_bundle["models"]
            mae_tr = last_mae_tr
            
    # Save model checkpoint and update pointer if retrained
    if reason is not None:
        m_filename = f"models_{origin_t.strftime('%Y%m%d')}.joblib"
        bundle = {
            "version": VERSION, "variant": BEST_VARIANT, "retrain_date": str(origin_t.date()),
            "reason": reason, "models": current_models, "mae_at_train": mae_tr,
            "feature_cols": FEATURE_COLS, "conformal_level": CONFORMAL_LEVEL
        }
        joblib.dump(bundle, OUT_DIR / m_filename)
        joblib.dump(bundle, LATEST_MODEL_POINTER)
        
        retrain_entry = pd.DataFrame([{
            "retrain_date": origin_t.date(), "reason": reason, "model_file": m_filename,
            **{f"mae_at_train_{h}w": mae_tr[h] for h in HORIZONS}
        }])
        r_log = pd.concat([r_log, retrain_entry], ignore_index=True).drop_duplicates(subset=["retrain_date"], keep="last")
        r_log.to_csv(RETRAIN_LOG_CSV, index=False)
        print(f"[RETRAIN] Executed ({reason}). Saved {m_filename} and updated {LATEST_MODEL_POINTER.name}")
    else:
        print(f"[NO RETRAIN] Weeks since last retrain: {weeks_since} (no drift detected)")
        
    # 4. Predict h = 1, 2, 4 weeks
    pct_now = float(feat_live.loc[origin_t, "pct_now"])
    X_now = feat_live.loc[[origin_t], FEATURE_COLS]
    new_forecast_rows = []
    latest_display = []
    
    for h in HORIZONS:
        target_dt = origin_t + pd.Timedelta(weeks=h)
        med_delta = float(predict_median(current_models[h]["median"], X_now)[0])
        lo_delta, hi_delta = apply_conformal(current_models[h]["conformal_offsets"], med_delta, pd.DatetimeIndex([target_dt]))
        seas_delta = float(seasonal_predict(current_models[h]["seasonal_table"], pd.DatetimeIndex([target_dt])).iloc[0])
        
        p_med = round(pct_now + med_delta, 2)
        p_lo = round(pct_now + float(lo_delta[0]), 2)
        p_hi = round(pct_now + float(hi_delta[0]), 2)
        p_seas = round(pct_now + seas_delta, 2)
        
        latest_display.append({
            "origin_week_end": origin_t.date(), "horizon_weeks": h, "target_week_end": target_dt.date(),
            "pct_full_now": round(pct_now, 2), "pred_low": p_lo, "pred_median": p_med, "pred_high": p_hi,
            "seasonal_baseline": p_seas
        })
        new_forecast_rows.append({
            "origin_date": origin_t.date(), "horizon": h, "target_date": target_dt.date(),
            "prediction": p_med, "low": p_lo, "high": p_hi, "seasonal_prediction": p_seas,
            "model_version": f"models_{r_log['retrain_date'].max().strftime('%Y%m%d')}",
            "retrain_reason": reason if reason else "", "actual": np.nan, "model_mae": np.nan,
            "seasonal_mae": np.nan, "in_band": np.nan
        })
        
    # 5. Save latest_forecast.csv & update forecast_log.csv
    latest_df = pd.DataFrame(latest_display)
    latest_df.to_csv(LATEST_FORECAST_CSV, index=False)
    
    f_log = pd.concat([f_log, pd.DataFrame(new_forecast_rows)], ignore_index=True).drop_duplicates(
        subset=["origin_date", "horizon"], keep="last")
    f_log.to_csv(FORECAST_LOG_CSV, index=False)
    
    return latest_df

# Execute the live job for latest Thursday
live_forecast = run_live_thursday_job()
live_forecast


[RETRAIN] Executed (initial). Saved models_20261022.joblib and updated models_latest.joblib


,origin_week_end,horizon_weeks,target_week_end,pct_full_now,pred_low,pred_median,pred_high,seasonal_baseline
0,2026-10-22,1,2026-10-29,93.51,85.60,92.49,101.49,92.64
1,2026-10-22,2,2026-11-05,93.51,86.57,90.36,92.08,92.31
2,2026-10-22,4,2026-11-19,93.51,80.20,86.25,90.33,91.31


[NO RETRAIN] Weeks since last retrain: 0 (no drift detected)


,origin_week_end,horizon_weeks,target_week_end,pct_full_now,pred_low,pred_median,pred_high,seasonal_baseline
0,2026-10-22,1,2026-10-29,93.51,85.60,92.49,101.49,92.64
1,2026-10-22,2,2026-11-05,93.51,86.57,90.36,92.08,92.31
2,2026-10-22,4,2026-11-19,93.51,80.20,86.25,90.33,91.31
